# Modelos de ensemble — custo, biblioteca escolhida e espaços de busca

**Cards 04A (#185) e 04B (#186).** As seções 1 a 5 são do #185 e a seção 6 é do #186, que chegou pela `develop` no mesmo `ensembles.ipynb`.

**Card 04A, issue #185.** Este notebook responde a uma pergunta só: o projeto deve usar o `HistGradientBoostingClassifier`, que já vem no scikit-learn, ou o `XGBClassifier`, que acrescenta uma dependência ao projeto e à sessão do Colab?

A pergunta não é de gosto. Os cards #186, #187 e #188 rodam uma busca aleatória de 40 iterações sobre os folds do contrato, e uma sessão do Colab não dura para sempre. Se um ajuste único custar tempo demais, a busca não cabe na sessão e os três cards seguintes travam. O que decide, então, é medição: quanto custa **um** ajuste completo de cada biblioteca sobre a matriz que o contrato entrega.

**O que este notebook não faz.** Ele não treina o modelo final, não roda busca de hiperparâmetros e não cria nenhuma partição ou validação cruzada própria. As partições são as do contrato do card 01 (`matriz.preparar_matriz`), e nenhuma outra: uma fatia aleatória criada aqui contrariaria o agrupamento por Cliente da seção 3 e ainda mediria um treino menor do que o real, devolvendo um tempo otimista.

## 1. Ambiente e versões

As três bibliotecas que entram na comparação estão fixadas em versão exata no `requirements.txt` desde o commit 1 deste card. O motivo é a própria medição: comparar tempo entre duas bibliotecas só tem sentido se a sessão do Colab e a máquina de quem revisa instalarem a mesma pilha. Com faixa (`>=`), duas execuções do mesmo notebook em dias diferentes podem medir versões diferentes e a diferença de tempo passaria a ser da versão, não da biblioteca.

A célula abaixo instala o `requirements.txt` e imprime as versões efetivamente carregadas.

In [ ]:
!pip install -q -r requirements.txt

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost

# Inclui as pastas de código tanto no projeto local quanto no Colab. O break
# evita que duas cópias do mesmo módulo fiquem no caminho de importação.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break
for pasta_scripts in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_scripts.exists():
        sys.path.insert(0, str(pasta_scripts.resolve()))
        break

import ensembles
importlib.reload(ensembles)

# Falha aqui, e nao tres secoes depois, se a sessao carregou uma versao
# diferente da fixada no requirements.txt (revisao da !94).
versoes_conferidas = ensembles.conferir_versoes()

print("scikit-learn", sklearn.__version__)
print("scipy       ", scipy.__version__)
print("xgboost     ", xgboost.__version__)
print("python      ", sys.version.split()[0])
print()
print("conferidas contra o requirements.txt:", versoes_conferidas)

O output acima é a evidência do CR01: as três versões impressas são exatamente as declaradas no `requirements.txt` (`scikit-learn==1.9.1`, `scipy==1.18.1`, `xgboost==3.4.1`), e a última linha mostra o que `ensembles.conferir_versoes` validou. A célula não depende de quem lê reparar numa divergência: se a sessão do Colab carregou uma cópia pré-instalada na frente do arquivo, `conferir_versoes` levanta `AssertionError` com o pacote, a versão esperada e a carregada, e o notebook para aqui. Nesse caso, reinicie o ambiente de execução depois do `pip install` e rode de novo.

A versão do scikit-learn passou de 1.9.0 para 1.9.1 ao incorporar a `develop`, que já fixava a 1.9.1 pelo #206; duas versões exatas da mesma biblioteca no projeto quebrariam a reprodutibilidade que as duas issues pedem.

## 2. Contrato de entrada (card 01)

Tudo o que a medição consome vem de `matriz.preparar_matriz`: as features (`x`), o alvo (`y`), o agrupamento por Cliente (`grupos`), as matrizes já transformadas e o `ColumnTransformer` ajustado **somente no treino**. Nada é reprocessado aqui.

Isso não é formalidade. O pré-processador do contrato imputa com mediana e indicador de ausência, escalona de forma robusta e codifica as categóricas em one-hot; um pré-processamento improvisado neste notebook produziria menos colunas e um ajuste mais rápido do que o que a busca do card #187 vai enfrentar de verdade. O número medido seria otimista justamente na direção que faria a decisão sair errada.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/data/processed/base_analitica.parquet"),
]
caminho = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Rode o pre-processamento.ipynb antes, ou "
        f"coloque o parquet em um destes caminhos: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho)

# Datas do registro de decisão da política de particionamento (#127/#128), as
# mesmas do modelagem.ipynb. Elas são parâmetro, nunca constante de módulo.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(df, CORTE_VALIDACAO, CORTE_TESTE)
matriz_treino = preparo["matrizes"]["treino"]
y_treino = preparo["y"]["treino"]

print(resumo_da_matriz(preparo))
print()
print("colunas da matriz:", preparo["metadados"]["colunas_da_matriz"])
print("Clientes distintos no treino:", preparo["grupos"]["treino"].nunique())

O output acima mostra as três partições com n, intervalo de datas e Clientes. Dois números importam para ler a medição da seção 3: o **n do treino**, porque é sobre ele que o ajuste acontece, e o **número de colunas da matriz**, porque num boosting por histograma o custo depende do número de colunas e de bins muito mais do que do número de linhas.

Repare que o treino é bem menor que a base inteira: o corte é temporal e por Cliente, então o treino é só o período mais antigo. Estimar o custo da busca a partir do total de registros da base superestimaria o tempo; o número que vale é o do treino.

## 3. Medição de custo de um ajuste

`ensembles.medir_bibliotecas` ajusta cada biblioteca **uma** vez sobre a matriz de treino do contrato e devolve tempo de ajuste, pico de memória do processo e a semente usada.

Três cuidados estão embutidos nessa função e são o que torna a comparação honesta:

1. **Hiperparâmetros equivalentes termo a termo.** Mesmo passo, mesmo número de árvores, mesmo número de folhas, mesmo número de bins, histograma nos dois lados. A célula abaixo usa `ensembles.HIPERPARAMETROS_COMPARACAO`, que **não é uma cópia**: passo, número de árvores e número de folhas são lidos de `modelo.HIPERPARAMETROS_CANDIDATO`, os hiperparâmetros do candidato que o projeto realmente treina. A célula imprime os dois lado a lado e falha se divergirem, e o teste `test_comparacao_usa_os_hiperparametros_do_candidato` trava a mesma igualdade fora do notebook.
2. **Sem parada antecipada nos dois lados.** Com ela ligada, o tempo medido seria o de menos árvores do que as declaradas — e, no caso do scikit-learn, a fatia de parada seria aleatória, ignorando o agrupamento por Cliente.
3. **Estimador clonado a cada medição.** Um modelo reaproveitado começaria de um ajuste pronto e pareceria mais barato.

O pico de memória sai do RSS do processo, amostrado em thread separada. Quase toda a alocação das duas bibliotecas é nativa, e `tracemalloc` enxerga só o lado Python: mediria alguns megabytes e diria que as duas custam o mesmo.

In [ ]:
import ensembles
importlib.reload(ensembles)
from ensembles import escolher_biblioteca, estimar_busca, medir_bibliotecas

SEMENTE = ensembles.SEMENTE_PADRAO

import modelo

candidato = modelo.HIPERPARAMETROS_CANDIDATO
print("hiperparâmetros da comparação:", ensembles.HIPERPARAMETROS_COMPARACAO)
print("hiperparâmetros do candidato: ",
      {k: candidato[k] for k in ("learning_rate", "max_iter", "max_leaf_nodes")})

# O tempo medido so e o do modelo treinado se os tres valores forem os mesmos.
assert ensembles.HIPERPARAMETROS_COMPARACAO["passo"] == candidato["learning_rate"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["arvores"] == candidato["max_iter"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["folhas"] == candidato["max_leaf_nodes"]

tabela_medicao = medir_bibliotecas(matriz_treino, y_treino, random_state=SEMENTE)
tabela_medicao[["tempo_ajuste_s", "pico_memoria_mb", "delta_memoria_mb", "random_state"]]

O output acima é a tabela do CR04. Ela tem uma linha por biblioteca, com o tempo de um ajuste completo em segundos, o pico de memória do processo em MB, o quanto desse pico foi acrescentado pelo próprio ajuste (`delta_memoria_mb`) e o `random_state` usado.

O `delta` está ali porque o pico sozinho engana: ele inclui a base carregada e a matriz transformada, que já estavam na memória antes do ajuste e são as mesmas para as duas bibliotecas. Quem diferencia uma da outra é o delta.

### 3.1. Medição registrada

Execução de 17/09/2026, semente 42, scikit-learn 1.9.0 (antes do alinhamento à 1.9.1 descrito na seção 1), sobre uma matriz de treino de 167.815 linhas e 31 colunas:

| Biblioteca | Tempo de um ajuste (s) | Pico de memória (MB) | Acréscimo do ajuste (MB) | `random_state` |
| --- | ---: | ---: | ---: | ---: |
| `HistGradientBoostingClassifier` | 3,58 | 582,4 | 33,7 | 42 |
| `XGBClassifier` | 2,65 | 567,4 | 22,5 | 42 |

**Ressalva sobre esta execução, que precisa ser lida antes de usar os números.** Não havia base da Azul na máquina onde a tabela acima foi medida, e o `.gitignore` do projeto proíbe versioná-la. Os números vieram de uma base **sintética com a mesma forma declarada na issue** — 484.915 registros e 407.139 valores distintos de `ID_GOLDENRECORD`, passando pelo mesmo `preparar_matriz` e produzindo as mesmas 31 colunas de matriz. O que isso mede bem é a ordem de grandeza e a razão entre as duas bibliotecas; o que isso não mede é o efeito da distribuição real das categorias sobre o número de bins efetivos.

Além disso, a medição rodou em máquina local com 14 CPUs lógicas e 15,5 GB de RAM. Uma sessão gratuita do Colab tem cerca de 2 vCPUs, e as duas bibliotecas paralelizam o ajuste: espere algo entre 5 e 7 vezes mais lento lá.

**Portanto:** ao rodar este notebook no Colab sobre a base real, substitua a tabela acima pelo output da célula da seção 3 e refaça a conta da seção 4. A decisão da seção 5 continua válida enquanto a conclusão da seção 4 continuar valendo — e a seção 4 diz explicitamente qual é o limite.

## 4. A busca dos cards seguintes cabe na sessão?

`estimar_busca` multiplica o tempo de um ajuste pelas 40 iterações da busca e pelo número de folds do contrato. A conta é deliberadamente otimista: conta só os ajustes e ignora transformação, cálculo de métrica e o fato de cada fold custar quase o treino inteiro. Serve como piso — se nem esse piso couber na sessão, a busca com certeza não cabe.

O orçamento de 3 horas é o que se assume disponível numa sessão de Colab antes de ela cair ou ser reciclada.

In [ ]:
ORCAMENTO_HORAS = 3.0
N_ITERACOES = 40
N_FOLDS = 5

for biblioteca, linha in tabela_medicao.iterrows():
    horas = estimar_busca(linha["tempo_ajuste_s"], N_ITERACOES, N_FOLDS)
    print(f"{biblioteca:32s} {horas:6.2f} h  ({N_ITERACOES} iterações x {N_FOLDS} folds)")

decisao = escolher_biblioteca(
    tabela_medicao,
    orcamento_horas=ORCAMENTO_HORAS,
    n_iteracoes=N_ITERACOES,
    n_folds=N_FOLDS,
)
print()
print("escolhida:", decisao["escolhida"])
print(decisao["justificativa"])

O output acima aplica o critério do CR05, e a ordem dele importa: primeiro descarta quem não cabe no orçamento da sessão, porque uma biblioteca que estoura a sessão não é uma opção mais lenta, é uma opção que não existe; entre as que cabem, vence a mais barata de manter.

Na execução registrada, as duas cabem com folga — 0,20 h para o `HistGradientBoostingClassifier` e 0,15 h para o `XGBClassifier`, contra um orçamento de 3 h. Mesmo aplicando o fator de 5 a 7 vezes do Colab, as duas seguem abaixo de 1,5 h. **O limite onde essa conclusão mudaria:** um ajuste precisaria custar mais de 54 segundos no ambiente onde a busca for rodar (3 h ÷ 200 ajustes). Se o output da sua execução passar disso, a decisão da seção 5 precisa ser refeita.

## 5. Decisão registrada

**O projeto usa o `HistGradientBoostingClassifier` como biblioteca de boosting dos modelos de ensemble.**

A justificativa tem duas partes, nesta ordem:

1. **Viabilidade.** As duas bibliotecas cabem no orçamento da sessão com margem larga. A busca de 40 iterações vezes 5 folds sai em 0,20 h com o `HistGradientBoostingClassifier` e 0,15 h com o `XGBClassifier` na medição registrada; mesmo no pior fator esperado do Colab, nenhuma das duas se aproxima das 3 h. Ou seja, o custo **não** é critério de desempate aqui — a diferença de 0,93 s por ajuste não muda o que é possível fazer.
2. **Custo de manutenção.** Como a viabilidade não separa as duas, decide o que sobra: o `HistGradientBoostingClassifier` já está no scikit-learn, que o projeto usa desde a seção 4.2.2, e não acrescenta dependência ao `requirements.txt` nem instalação à sessão do Colab. O `XGBClassifier` acrescenta as duas coisas em troca de 26% de tempo num ajuste que já custa segundos.

**O que faria esta decisão ser revista.** O `XGBClassifier` foi medido mais rápido, e a vantagem dele cresce com o tamanho do problema. Se um card futuro aumentar a matriz — mais features, mais categorias, mais bins — a ponto de um ajuste passar dos 54 segundos no ambiente de execução, a ordem do critério se inverte e a dependência extra passa a se pagar. Por isso o `xgboost` fica declarado no `requirements.txt` mesmo não sendo o escolhido agora: a medição precisa continuar reproduzível para quem refizer essa conta.

**Comunicação.** Esta decisão vale para os cards #186, #187 e #188 e precisa chegar à dupla de Métricas e Decisões, que depende dela para dimensionar a busca.

## 6. Espaços de busca da busca aleatória (card 04B, #186)

As seções 1 a 5 mediram **quanto custa** um ajuste e escolheram a biblioteca de boosting. Esta seção declara **em que intervalo** cada hiperparâmetro pode variar na busca dos cards seguintes, e é o conteúdo que veio do #186, com a numeração ajustada para caber neste notebook.

Este notebook documenta e justifica os espaços de busca aleatória (`RandomizedSearchCV`) do Random Forest e do Gradient Boosting, definidos em `src/ensembles.py`. Ele **não executa nenhuma busca**: treinar as combinações nos folds do #102 e comparar contra os pisos fica para os cards que dependem deste — #187 (Random Forest), #188 e #189 (Gradient Boosting) e #190.

**Pontos de partida em comum com o candidato único da seção 4.3.** A base analítica tem 484.915 registros e 20,44% de Detratores (seção 4.2.1 da documentação); o treino usado pela validação cruzada por Cliente do #102, depois do split, tem 341.962 linhas. O contrato de features do card 01 (`FEATURE_SET_V1`, em `scripts/preprocessamento_nps.py`) declara 11 features brutas, e é o mesmo contrato usado aqui.

**Gradient Boosting: duas bibliotecas, por enquanto.** O #185 ainda não decidiu entre `HistGradientBoostingClassifier` (scikit-learn) e `XGBClassifier` (XGBoost) para o trabalho de Ensemble, então este notebook e `src/ensembles.py` mantêm os dois espaços lado a lado. No dia em que o #185 decidir, a biblioteca descartada sai dos dois lugares no mesmo commit.

**Os testes que garantem que os espaços funcionam** ficam em `tests/test_ensembles.py`, e não neste notebook: eles sorteiam 40 combinações de cada espaço com semente fixa e instanciam o estimador correspondente, o que é rápido demais para justificar rodar aqui a cada execução do notebook. Este notebook sorteia só uma amostra pequena, para inspeção visual dos valores.

### 6.1. Random Forest

#### 6.1.1. Espaço de busca declarado

A célula abaixo importa `ESPACO_RANDOM_FOREST` de `src/ensembles.py` e lista os cinco eixos declarados. A importação do módulo não ajusta nenhum estimador (CR01): o que a célula imprime é a distribuição em si, não uma amostra dela.

In [ ]:
# O caminho de importacao ja foi preparado na secao 1.
from ensembles import ESPACO_RANDOM_FOREST


def descrever(distribuicao):
    """Nome e parâmetros da distribuição, em vez do endereço de memória do repr padrão.

    O repr padrão de um objeto `scipy.stats` traz o endereço em memória, que muda
    a cada execução e sujaria o diff do notebook sem nenhuma informação nova.
    """
    if isinstance(distribuicao, list):
        return f"lista com {len(distribuicao)} opções: {distribuicao}"
    return f"scipy.stats.{distribuicao.dist.name}, parâmetros={distribuicao.args or distribuicao.kwds}"


for nome, distribuicao in ESPACO_RANDOM_FOREST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `n_estimators`, `max_depth` e `min_samples_leaf` são inteiros sorteados por `scipy.stats.randint`; `max_features` é uma fração contínua sorteada por `scipy.stats.uniform`; `class_weight` é a única lista simples do espaço, porque reponderação de classe é uma escolha categórica, sem escala, e não uma distribuição numérica. A tabela consolidada da seção 3 traz o intervalo e a justificativa de cada um.

#### 6.1.2. Amostra de combinações, para inspeção visual

`ParameterSampler` sorteia cinco combinações do espaço com semente fixa. Cinco é o suficiente para olhar os valores a olho nu; a garantia de que 40 combinações são aceitas pelo `RandomForestClassifier` é o que `tests/test_ensembles.py` cobre, não esta célula.

In [ ]:
import pandas as pd
from sklearn.model_selection import ParameterSampler

SEMENTE = 42

amostra_rf = pd.DataFrame(
    ParameterSampler(ESPACO_RANDOM_FOREST, n_iter=5, random_state=SEMENTE)
)
amostra_rf

**Leitura da tabela.** Cada linha é uma combinação que um `RandomizedSearchCV` real sortearia e passaria ao `RandomForestClassifier`. `max_features` sai como fração entre 0,3 e 1,0, nunca uma contagem de colunas; `class_weight` alterna entre as três opções declaradas. Os valores concretos mudam se a semente mudar, mas a mesma semente reproduz sempre esta tabela, como `test_amostragem_e_reprodutivel_com_a_mesma_semente` confere.

### 6.2. Gradient Boosting

Duas bibliotecas convivem aqui até o #185 decidir. As duas seções abaixo seguem a mesma estrutura da seção 1: espaço declarado, depois amostra pequena para inspeção.

#### 6.2.1. `HistGradientBoostingClassifier` (scikit-learn)

Mesma biblioteca do candidato único da seção 4.3 do notebook `modelagem.ipynb`. A célula lista `ESPACO_GRADIENT_BOOSTING_HISTGB`.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_HISTGB.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `learning_rate` é log-uniforme, porque o efeito do passo sobre o número de iterações necessário é multiplicativo; `max_leaf_nodes`, `max_iter` e `min_samples_leaf` são inteiros; `class_weight` é lista, pelo mesmo motivo da seção 1. `early_stopping` não entra no espaço — continua fixo em `False` em `criar_candidato` (`src/modelo.py`), porque ligá-lo vazaria Cliente entre ajuste e medição interna, e isso não é uma escolha que a busca deva reabrir.

#### 6.2.2. `XGBClassifier` (XGBoost)

A célula lista `ESPACO_GRADIENT_BOOSTING_XGBOOST`. Sete eixos, dois a mais que o espaço do HistGB, porque o XGBoost expõe sub-amostragem de linhas e de colunas como parâmetros próprios, que o HistGB não tem por discretizar em histogramas em vez de amostrar.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_XGBOOST

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_XGBOOST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** `learning_rate` usa a mesma distribuição log-uniforme do HistGB, para o custo por ajuste do #185 comparar as duas bibliotecas sob o mesmo raciocínio de passo. `scale_pos_weight` é o único eixo com teto calculado a partir da base: `RAZAO_DESBALANCEAMENTO` (~3,89) é a proporção de não-Detratores por Detrator, e um `scale_pos_weight` nesse valor neutraliza o desbalanceamento por completo.

#### 6.2.3. Amostra combinada, para inspeção visual

Uma amostra pequena de cada biblioteca, lado a lado.

In [ ]:
amostra_histgb = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_HISTGB, n_iter=5, random_state=SEMENTE)
)
amostra_xgboost = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_XGBOOST, n_iter=5, random_state=SEMENTE)
)
print("HistGradientBoostingClassifier:")
display(amostra_histgb)
print("XGBClassifier:")
display(amostra_xgboost)

**Leitura das tabelas.** As duas amostram a mesma semente e o mesmo `n_iter`, mas eixos diferentes: `max_leaf_nodes` de um lado, `max_depth` do outro, porque cada biblioteca limita o tamanho da árvore à sua maneira. Nenhuma das duas tabelas depende da outra ter rodado antes — os dois espaços são independentes, e é isso que permite descartar um deles no dia em que o #185 decidir sem tocar no que sobra.

### 6.3. Tabela consolidada de hiperparâmetros (CR04)

| Modelo | Hiperparâmetro | Distribuição | Intervalo | Justificativa |
|---|---|---|---|---|
| Random Forest | `n_estimators` | `randint` | 200 a 600 | Com 341.962 linhas de treino a variância de cada árvore já é pequena; acima de algumas centenas o ganho de agregar mais árvores fica marginal e o custo segue linear no número delas. |
| Random Forest | `max_depth` | `randint` | 3 a 20 | Teto próximo de log₂(341.962) ≈ 18,4: uma árvore balanceada raramente precisa de mais níveis para separar o treino inteiro. Profundidade ilimitada arriscaria árvores memorizando Cliente, o mesmo risco documentado para `min_samples_leaf` no #103. |
| Random Forest | `min_samples_leaf` | `randint` | 1 a 100 | 100 é o valor que o #103 escolheu para o Gradient Boosting nesta mesma base; o piso em 1 mantém a ponta não regularizada disponível para a busca comparar. |
| Random Forest | `max_features` | `uniform` | 0,3 a 1,0 (fração) | O contrato do card 01 declara 11 features brutas; `"sqrt"`/`"log2"` datam de bases com centenas de colunas e amostrariam ~3 delas por split, descartando a maior parte do sinal disponível. |
| Random Forest | `class_weight` | lista | `balanced`, `balanced_subsample`, `None` | 20,44% de detração é desbalanceamento moderado. Ao contrário do candidato único (onde reponderar prejudicaria a probabilidade calibrada do #103), o Random Forest vota por árvore e não expõe a mesma dependência direta da calibração — a busca decide empiricamente qual das três funciona melhor nesta base. |
| Gradient Boosting (HistGB) | `learning_rate` | `loguniform` | 0,01 a 0,3 | O #103 fixou 0,05 e a grade do #104 testou 0,05/0,10; escala log porque o efeito do passo sobre o número de iterações necessário é multiplicativo. |
| Gradient Boosting (HistGB) | `max_leaf_nodes` | `randint` | 15 a 127 | Cobre o padrão da biblioteca (31, usado pelo #103) e o ponto testado pela grade do #104 (63), com margem para os dois lados. |
| Gradient Boosting (HistGB) | `max_iter` | `randint` | 100 a 600 | Cobre os três pontos da grade do #104 (150, 300, 600). Sem parada antecipada, é o único limite do ensemble. |
| Gradient Boosting (HistGB) | `l2_regularization` | `uniform` | 0,0 a 2,0 | O #103 fixou 1,0 contra o padrão 0,0 da biblioteca; o intervalo cobre os dois extremos e o dobro da escolha do #103. |
| Gradient Boosting (HistGB) | `min_samples_leaf` | `randint` | 20 a 200 | Mesmo raciocínio do Random Forest: folha pequena sobre 341.962 linhas memoriza Cliente. O piso de 20 é o padrão da biblioteca. |
| Gradient Boosting (HistGB) | `class_weight` | lista | `None`, `balanced` | O #103 documenta por que `class_weight=None` foi a escolha do candidato: reponderar deslocaria a probabilidade predita, piorando o Brier da seção 6 e o limiar da seção 7. O eixo entra na busca para que a comparação empírica sustente essa escolha, não só o argumento teórico. |
| Gradient Boosting (XGBoost) | `learning_rate` | `loguniform` | 0,01 a 0,3 | Mesma faixa e mesma razão log-uniforme do HistGB, por serem as duas gradient boosting sobre árvores. |
| Gradient Boosting (XGBoost) | `max_depth` | `randint` | 3 a 10 | O XGBoost limita a árvore por profundidade, e não por número de folhas; 10 níveis já produzem até 2¹⁰ folhas, ordem de interação comparável ao teto de 127 folhas do espaço irmão. |
| Gradient Boosting (XGBoost) | `n_estimators` | `randint` | 100 a 600 | Mesmo intervalo de `max_iter` do HistGB, para que o custo por ajuste medido no #185 seja comparável entre as duas bibliotecas sob o mesmo orçamento de iterações. |
| Gradient Boosting (XGBoost) | `min_child_weight` | `randint` | 1 a 20 | Equivalente do XGBoost a `min_samples_leaf`: soma mínima de peso das observações numa folha antes dela poder ser criada, mesma lógica de regularização contra folha memorizando Cliente. |
| Gradient Boosting (XGBoost) | `reg_lambda` | `uniform` | 0,0 a 2,0 | Mesma faixa de `l2_regularization` do HistGB, pelo nome equivalente no XGBoost. |
| Gradient Boosting (XGBoost) | `subsample` | `uniform` | 0,6 a 1,0 | Mecanismo de regularização do XGBoost sem par direto no HistGB (que discretiza em histogramas em vez de amostrar linhas). O piso de 0,6 evita perder informação suficiente para exigir muito mais iterações. |
| Gradient Boosting (XGBoost) | `colsample_bytree` | `uniform` | 0,5 a 1,0 | Equivalente a `max_features` do Random Forest: com 11 features brutas no contrato do card 01, um piso baixo deixaria cada árvore enxergar poucas colunas. |
| Gradient Boosting (XGBoost) | `scale_pos_weight` | `uniform` | 1,0 a ~3,89 | O teto é `RAZAO_DESBALANCEAMENTO`, a proporção observada de não-Detratores por Detrator (385.775 / 99.140). O XGBoost não tem uma versão `"balanced"` pronta como o scikit-learn; este parâmetro multiplica o gradiente da classe positiva, e o teto no valor exato da proporção é o ponto em que a reponderação neutraliza o desbalanceamento por completo. |

**O que fica para os cards seguintes.** Este notebook não mede custo por ajuste nem escolhe `n_iter` para a `RandomizedSearchCV` real — isso depende do #185, que ainda está aberto. Quando ele decidir a biblioteca de Gradient Boosting, a seção 2 deste notebook e o espaço correspondente em `src/ensembles.py` perdem a metade descartada no mesmo commit.

## 7. Linha de base do Random Forest (card 08A, #187)

**Card 08A, issue #187.** Antes de a busca do #189 variar qualquer hiperparâmetro, o Random Forest precisa de um ponto de partida medido: sem ele, o ganho que a busca reportar não tem contra o que ser lido, e qualquer número que ela produzir parece bom.

O modelo não entra solto. `ensembles.criar_pipeline_random_forest` encadeia num único `Pipeline` o `ColumnTransformer` do contrato, `preparo["preprocessador"]` da seção 2, e um `RandomForestClassifier`. O pré-processador entra **clonado**: mesma especificação do contrato, sem o ajuste que `preparar_matriz` já fez no treino. É isso que permite à busca do #189 reajustar imputação e escala dentro de cada fold, em vez de herdar medianas que já viram as linhas de validação.

Duas decisões desta seção, declaradas aqui para não ficarem implícitas:

1. **Hiperparâmetros no padrão da biblioteca.** A linha de base não escolhe profundidade, número de árvores nem `class_weight`; quem escolhe é a busca.
2. **`random_state` = `SEMENTE` (42), a mesma da seção 3.** O Random Forest sorteia a amostra de cada árvore e as colunas de cada divisão; sem semente fixa, dois ajustes idênticos dariam florestas diferentes e a diferença de métrica na busca seria indistinguível de sorte.

A célula abaixo monta o pipeline e imprime os passos e os parâmetros do estimador, sem ajustar nada ainda.

In [ ]:
from ensembles import PASSO_MODELO, PASSO_PREPARO, criar_pipeline_random_forest

pipeline_rf = criar_pipeline_random_forest(preparo["preprocessador"], random_state=SEMENTE)

print("passos:", [nome for nome, _ in pipeline_rf.steps])
print("pré-processador é o objeto do contrato?", pipeline_rf.named_steps[PASSO_PREPARO] is preparo["preprocessador"])
print()
parametros_rf = pipeline_rf.named_steps[PASSO_MODELO].get_params()
for chave in ("n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight", "random_state"):
    print(f"{chave:18s} {parametros_rf[chave]}")

O output acima tem três leituras. Os passos precisam ser `['preparo', 'modelo']`, os mesmos nomes do pipeline da logística (#208), porque é por `passo__parametro` que as grades das duplas são endereçadas. A pergunta sobre o objeto do contrato precisa responder `False`: o pipeline trabalha sobre uma cópia virgem, e o `preparo["preprocessador"]` usado nas seções anteriores continua intacto. E os parâmetros impressos precisam ser os padrões do scikit-learn (100 árvores, profundidade livre, `max_features='sqrt'`, sem `class_weight`), com `random_state` igual a 42.

### 7.1. Medição da linha de base

`ensembles.medir_linha_de_base` ajusta o pipeline sobre a partição de treino **crua**, `preparo["x"]["treino"]`, pontua a partição de validação e entrega `y_true`, `y_pred` e `y_proba` à função `avaliar` do card 05 (#241). Nenhuma métrica é calculada aqui e nenhum particionador é instanciado: o número que sai é o que `avaliar` devolver, com os mesmos nomes de métrica que as outras duplas vão reportar.

A matriz crua é deliberada. Passar `preparo["matrizes"]["treino"]`, que já saiu do pré-processador, faria o `ColumnTransformer` do pipeline ser ajustado sobre a saída de outro, sem erro nenhum e com colunas que a base não tem.

**Enquanto o #241 não estiver em `develop`, a célula abaixo não mede nada** e diz isso no output. É o caminho que a própria issue #187 prevê para esse atraso, e ela proíbe explicitamente contorná-lo com validação cruzada própria. Quando o #241 chegar, basta ajustar `MODULO_AVALIAR` ao nome do módulo publicado e reexecutar.

In [ ]:
from ensembles import medir_linha_de_base

MODULO_AVALIAR = "avaliacao"  # ajustar ao módulo que o card 05 (#241) publicar em src/

try:
    avaliar = getattr(importlib.import_module(MODULO_AVALIAR), "avaliar")
except (ImportError, AttributeError):
    avaliar = None

if avaliar is None:
    linha_de_base_rf = None
    print(f"avaliar() ainda não está disponível em src/{MODULO_AVALIAR}.py (card 05, #241).")
    print("Linha de base não medida; nenhuma métrica substituta foi calculada.")
else:
    linha_de_base_rf = medir_linha_de_base(
        pipeline_rf,
        preparo["x"]["treino"],
        preparo["y"]["treino"],
        preparo["x"]["validacao"],
        preparo["y"]["validacao"],
        avaliar=avaliar,
    )
    print("random_state:", linha_de_base_rf["random_state"])
    print(f"treino: {linha_de_base_rf['n_treino']} linhas | validação: {linha_de_base_rf['n_avaliacao']} linhas")
    print(f"tempo total (ajuste + pontuação): {linha_de_base_rf['tempo_total_s']:.1f} s")
    print()
    for nome, valor in linha_de_base_rf["metricas"].items():
        print(f"{nome:28s} {valor:.4f}")

O output acima tem uma de duas formas. Se `avaliar` ainda não existe, ele diz que a linha de base não foi medida, e é isso que deve ficar registrado: um número calculado por métrica própria entraria na tabela comparativa do card 18A.1 sem ser comparável com os das outras duplas. Se `avaliar` existe, ele imprime a semente, o tamanho das duas partições, o tempo total de ajuste e pontuação e as métricas exatamente com os nomes que `avaliar` devolveu (F2, Sensibilidade, Precisão Média e ROC-AUC, pelo #241).

O tempo total importa para o #189 do mesmo jeito que o da seção 3 importou para a escolha da biblioteca: multiplicado pelas iterações e pelos folds da busca, ele diz se a busca do Random Forest cabe na sessão do Colab.

### 7.2. Linha de base registrada

| Campo | Valor |
| --- | --- |
| `random_state` | 42 |
| Hiperparâmetros | padrão do scikit-learn 1.9.1, a versão fixada no `requirements.txt` |
| Partição medida | validação do contrato (de `2025-07-01` a `2026-01-01`) |
| Folds | nenhum: `avaliar(y_true, y_pred, y_proba)` do #241 recebe vetores e não particiona |
| Métricas de `avaliar` | **pendente do #241** |
| Tempo total | **pendente da execução no Colab sobre a base real** |

**Ressalva.** A issue #187 descreve `avaliar(modelo, X, y, groups)` e pede o número de folds informado por ela. O card dono da função, o #241, define outra assinatura, `avaliar(y_true, y_pred, y_proba)`, que não recebe modelo nem particiona; o pipeline segue a do #241, a mesma que o pipeline da logística (#208) já usa. Por isso a linha de base é medida na partição de validação, e a medida por fold fica para a busca do #189, que usa os folds do contrato.

## 8. Busca aleatória do Random Forest (card 08B, #189)

**Card 08B, issue #189.** A seção 7 deixou o Random Forest montado sobre o pré-processador do contrato, com os hiperparâmetros padrão da biblioteca. Esta seção entrega o pipeline a uma `RandomizedSearchCV`, que varre o espaço declarado pelo #186 e devolve os hiperparâmetros vencedores num arquivo que qualquer pessoa consegue usar para remontar o modelo.

Três travas valem para tudo o que segue, e nenhuma delas é escolha desta seção:

1. **A validação é agrupada por Cliente.** A base tem 407.139 `ID_GOLDENRECORD` em 484.915 respostas, então o mesmo Cliente aparece em vários voos. Os folds vêm de `validacao.criar_folds`, conferidos por `validacao.conferir_folds` antes da busca, e o `fit` recebe `groups`. `busca_random_forest.criar_busca_random_forest` recusa `cv` inteiro: num classificador, o scikit-learn converteria o inteiro num particionador estratificado que ignora o Cliente, e o vencedor seria o que mais memoriza pessoas.
2. **`n_iter = 40` e `random_state = 42`.** Quarenta é o mínimo do card, e a função recusa menos. A semente fixa ao mesmo tempo o sorteio das combinações e a semente de cada floresta, então duas execuções desta seção sorteiam as mesmas 40 combinações e elegem o mesmo vencedor.
3. **A partição de teste não é tocada.** A busca recebe só `preparo["x"]["treino"]`, e a reconstrução da seção 8.3 é medida na validação.

O espaço é o `ESPACO_RANDOM_FOREST` do #186, e o `scoring` é o `scorer_f2` do #242, o critério de busca que o protocolo do #238 escolheu. Enquanto algum dos dois não estiver em `develop`, as células abaixo dizem isso no output e não rodam a busca com um substituto: um vencedor escolhido por outro espaço ou outro critério não serviria à tabela comparativa do card 18A.1.

In [ ]:
import busca_random_forest as busca_rf
importlib.reload(busca_rf)
from validacao import N_FOLDS, conferir_folds, criar_folds

N_ITER = busca_rf.N_ITER_MINIMO

try:
    from ensembles import ESPACO_RANDOM_FOREST  # card #186
except ImportError:
    ESPACO_RANDOM_FOREST = None
try:
    from scorer_f2 import scorer_f2  # card #242
except ImportError:
    scorer_f2 = None

pendentes = [nome for nome, objeto in (("#186 (ESPACO_RANDOM_FOREST)", ESPACO_RANDOM_FOREST),
                                        ("#242 (scorer_f2)", scorer_f2)) if objeto is None]

# Os folds vêm do contrato, agrupados por Cliente, e são conferidos antes de tudo.
folds_rf = criar_folds(preparo["x"]["treino"], preparo["grupos"]["treino"])
conferir_folds(folds_rf, preparo["grupos"]["treino"])
print(f"folds do contrato: {len(folds_rf)}, conferidos contra o Cliente")

if pendentes:
    busca = None
    print("busca não montada; falta em develop:", ", ".join(pendentes))
else:
    busca = busca_rf.criar_busca_random_forest(
        preparo["preprocessador"], ESPACO_RANDOM_FOREST, folds_rf, scorer_f2,
        n_iter=N_ITER, random_state=SEMENTE, n_jobs=-1,
    )
    print(f"n_iter={busca.n_iter}, random_state={busca.random_state}, folds={len(busca.cv)}")
    print(f"ajustes previstos: {N_ITER} x {N_FOLDS} + refit = {N_ITER * N_FOLDS + 1}")

O output acima tem duas partes. A primeira linha confirma que os cinco folds vieram de `criar_folds` e passaram por `conferir_folds`: nenhum Cliente aparece em duas validações, nem no ajuste e na validação do mesmo fold. A segunda mostra ou a configuração da busca, com `n_iter=40`, `random_state=42`, 5 folds e 201 ajustes previstos, ou a lista do que falta em `develop` para montá-la.

`n_jobs=-1` distribui os ajustes inteiros entre os núcleos da sessão. Cada floresta roda com `n_jobs=1`, como o #187 fixou, porque threads dentro dela mudariam o último bit de `predict_proba` e, num empate em 0,5, o rótulo; entre processos, a ordem em que os ajustes terminam não muda nenhum número.

### 8.1. Execução da busca e artefatos versionados

`busca_random_forest.executar_busca` confere os folds de novo, ajusta a busca sobre a matriz **crua** do treino (é o pipeline que transforma, dentro de cada fold) e mede o tempo total. `salvar_resultados` grava dois arquivos em `assets/`, os únicos desta seção que vão para o git:

- `hiperparametros_random_forest.json`: os hiperparâmetros vencedores, o `random_state`, o `n_iter`, o número de folds, o melhor F2 médio e o tempo total;
- `cv_resultados_random_forest.json`: uma linha por combinação, só com parâmetros, F2 médio, desvio entre folds, posição no ranking e tempo médio de ajuste. Nenhum dado de Cliente.

O resumo é JSON, e não o CSV que a issue pede, porque o `.gitignore` do projeto proíbe `*.csv` por compromisso firmado com o parceiro. O modelo ajustado não é versionado, por tamanho: ele é remontado a partir do JSON na seção 8.2.

In [ ]:
if busca is None:
    relato_busca = None
    print("busca não executada: ver o output da célula anterior.")
else:
    relato_busca = busca_rf.executar_busca(
        busca, preparo["x"]["treino"], preparo["y"]["treino"], preparo["grupos"]["treino"],
    )
    caminho_hp, caminho_resumo = busca_rf.salvar_resultados(busca, relato_busca, random_state=SEMENTE)

    vencedores_rf = busca_rf.hiperparametros_vencedores(busca)
    print("hiperparâmetros vencedores:")
    for nome, valor in vencedores_rf.items():
        print(f"  {nome:18s} {valor}")
    print(f"\nmelhor F2 médio nos folds: {relato_busca['melhor_score_medio']:.4f}")
    print(f"tempo total da busca: {relato_busca['tempo_total_s']:.0f} s "
          f"({relato_busca['tempo_total_s'] / 60:.1f} min), {relato_busca['n_ajustes']} ajustes")
    print("no limite do intervalo:", busca_rf.parametros_no_limite(vencedores_rf, ESPACO_RANDOM_FOREST) or "nenhum")
    print("\ngravados:", caminho_hp.name, "e", caminho_resumo.name)
    display(busca_rf.resumir_cv_results(busca).head(10))

O output acima traz, quando a busca roda, os cinco hiperparâmetros vencedores, o melhor F2 médio nos cinco folds, o tempo total e o número de ajustes, que precisa ser 201 (40 combinações vezes 5 folds, mais o refit): menos do que isso significaria fold pulado ou combinação que falhou. A linha "no limite do intervalo" lista os vencedores que caíram na borda do espaço do #186. Um `max_depth` vencedor igual a 20, o teto, diria que a busca queria ir além, e isso tem de ser comentado na tabela da seção 8.3, como pede o "Como revisar" do card. A tabela no fim mostra as dez melhores combinações, com o desvio entre folds ao lado da média: uma diferença de F2 menor do que esse desvio não separa dois candidatos.

### 8.2. Reconstrução pelo JSON e número oficial (CR04)

O F2 médio da seção 8.1 é o critério da busca, não o número que vai para a tabela comparativa. O número oficial é o de `avaliar`, do card 05A.1 (#241), e ele precisa sair de um modelo que qualquer pessoa consiga remontar sem esta sessão aberta. A célula abaixo lê o JSON gravado, remonta o pipeline com `busca_rf.reconstruir_pipeline` (o mesmo `criar_pipeline_random_forest` do #187, com a mesma semente), ajusta no treino e mede na validação com `avaliar`. Quando a busca rodou nesta sessão, a célula também mede o `best_estimator_` e exige que os dois números sejam iguais.

In [ ]:
try:
    avaliar = getattr(importlib.import_module("avaliacao"), "avaliar")  # card #241
except (ImportError, AttributeError):
    avaliar = None

if not busca_rf.ARQUIVO_HIPERPARAMETROS.exists():
    metricas_rf = None
    print(f"{busca_rf.ARQUIVO_HIPERPARAMETROS.name} ainda não existe: a busca da seção 8.1 não rodou.")
elif avaliar is None:
    metricas_rf = None
    print("avaliar() ainda não está em develop (card 05A.1, #241); nenhuma métrica substituta foi calculada.")
else:
    registro_rf = busca_rf.carregar_hiperparametros()
    reconstruido_rf = busca_rf.reconstruir_pipeline(preparo["preprocessador"], registro_rf)
    reconstruido_rf.fit(preparo["x"]["treino"], preparo["y"]["treino"])

    x_val, y_val = preparo["x"]["validacao"], preparo["y"]["validacao"]
    metricas_rf = avaliar(y_val, reconstruido_rf.predict(x_val), reconstruido_rf.predict_proba(x_val)[:, 1])
    print("reconstruído do JSON, medido na validação:")
    for nome, valor in metricas_rf.items():
        print(f"  {nome:16s} {valor:.4f}")

    if busca is not None:
        melhor = busca.best_estimator_
        metricas_busca = avaliar(y_val, melhor.predict(x_val), melhor.predict_proba(x_val)[:, 1])
        assert metricas_busca == metricas_rf, "o JSON não reconstrói o vencedor da busca"
        print("\nidêntico ao best_estimator_ da busca desta sessão: sim")

O output acima tem uma de três formas: o JSON ainda não existe, `avaliar` ainda não está em `develop`, ou as quatro métricas do protocolo (F2, Sensibilidade, Precisão Média e ROC-AUC) do modelo reconstruído, medidas na validação. Na terceira forma, se a busca rodou nesta sessão, a última linha confirma que o reconstruído e o `best_estimator_` dão exatamente o mesmo número. É essa igualdade que faz do JSON versionado uma descrição completa do vencedor, e é o número impresso aqui que vai para a tabela da seção 8.3. A igualdade é exata, e não aproximada, por causa do `n_jobs=1` da floresta.

### 8.3. Espaço de busca, vencedores e comparação com a linha de base

| Hiperparâmetro | Espaço do #186 | Vencedor | No limite? |
| --- | --- | --- | --- |
| `n_estimators` | inteiro uniforme de 200 a 600 | **pendente** | — |
| `max_depth` | inteiro uniforme de 3 a 20 | **pendente** | — |
| `min_samples_leaf` | inteiro uniforme de 1 a 100 | **pendente** | — |
| `max_features` | fração uniforme de 0,3 a 1,0 | **pendente** | — |
| `class_weight` | `balanced`, `balanced_subsample` ou `None` | **pendente** | não se aplica |

| Registro | Valor |
| --- | --- |
| `n_iter` | 40 |
| `random_state` | 42 |
| Folds | 5, de `validacao.criar_folds`, agrupados por Cliente |
| F2 médio na validação cruzada (5 folds) do vencedor | **pendente da execução** |
| Tempo total da busca | **pendente da execução no Colab** |
| Métricas de `avaliar` do vencedor (validação) | **pendentes do #241 e da execução** |
| Linha de base do #187 (seção 7.2) | **pendente do #241** |

**Por que a tabela está pendente.** A busca precisa de três peças que ainda não estão em `develop`: o espaço do #186 (!96), o `scorer_f2` do #242 (!104) e `avaliar` do #241 (!108). Além disso, não há base da Azul na máquina onde esta seção foi escrita. As células das seções 8 a 8.2 foram escritas para rodar sem alteração assim que as três MRs entrarem: rodá-las no Colab sobre a base real preenche esta tabela e grava os dois JSON de `assets/`. Nenhum número acima foi estimado ou inventado.

**Comparação com a linha de base (CR05).** Quando as duas medições existirem, a leitura é a diferença entre as métricas de `avaliar` do vencedor e as da linha de base da seção 7, na mesma partição de validação, com o tempo total da busca ao lado: um ganho de F2 pequeno diante de horas de busca é informação para o card 18A.1, não um detalhe.

**Onde estes números aparecem na documentação.** As duas tabelas desta seção são as que a Seção 4.4.4 da documentação (`documents/documentacao.md`, subseção "Hiperparâmetros vencedores e métricas") reproduz. Ao preencher os campos pendentes com os outputs das células das seções 7.1, 8.1 e 8.2, atualize a documentação no mesmo commit, para que cada número do texto continue idêntico ao output da célula que o gera. Antes de rodar, a seção 2 precisa usar os cortes do registro de decisão, `2025-07-01` e `2026-01-01`, os mesmos das seções 10 e 11.

## 9. Permutation importance do melhor ensemble (card 14A, #191)

**Card 14A, issue #191.** As seções 7 e 8 dizem **quanto** o Random Forest acerta. Esta seção responde a outra pergunta, que é a que a Azul faz primeiro: **em que o modelo se apoia para prever detração?** A resposta precisa ser dada em termos de fatores operacionais que a companhia reconhece, como atraso, número de trechos e cancelamento, e não em termos de colunas internas do pipeline.

O método é a *permutation importance*. Para cada feature, embaralham-se os valores dela na partição de avaliação, desfazendo a relação com o alvo e mantendo todo o resto intacto, e mede-se quanto a métrica cai. Uma queda grande diz que o modelo depende daquela feature para acertar; uma queda nula diz que ele passaria bem sem ela. O grupo decidiu não usar SHAP, e a permutation importance tem duas vantagens para este uso: não depende do tipo de modelo, o que vale igualmente para o Random Forest e para o Gradient Boosting, e é expressa na mesma métrica que escolheu o modelo.

Quatro travas valem para tudo o que segue, e todas estão em `src/explicabilidade.py`:

1. **A importância é das features originais do contrato.** O `ColumnTransformer` abre cada categórica em várias colunas one-hot. Medir depois dele espalharia `TIER_VIAGEM` em uma coluna por tier, e o ranking compararia pedaços de feature com features inteiras. Por isso a função recebe o **pipeline inteiro** e a matriz **crua**: a coluna é embaralhada antes do pré-processador, que roda depois, dentro do pipeline. Uma matriz já transformada é recusada.
2. **A partição é a de avaliação, nunca a de treino.** No treino, uma feature que o modelo memorizou parece importante. A partição é lida pelo nome do contrato, e `"treino"` é recusado pela função.
3. **O `scoring` é o mesmo da busca**, o `scorer_f2` do #242. Explicar o modelo por uma métrica diferente da que o escolheu misturaria duas perguntas.
4. **`n_repeats` e `random_state` são fixos.** Sem semente, dois embaralhamentos dão quedas diferentes, e um empate entre a terceira e a quarta feature trocaria o ranking de uma execução para outra.

**O que esta seção não afirma.** Importância por permutação mede o quanto o modelo **usa** uma feature, e não o quanto ela **causa** detração. A seção 9.5 volta a esse ponto.

### 9.1. Escolha do melhor ensemble

O melhor ensemble é o de maior **F2** medido por `avaliar` (#241) na partição de validação. O F2 é o critério porque foi ele que as duas buscas otimizaram (#242) e é ele o `scoring` da permutation importance: escolher o modelo por uma métrica e explicá-lo por outra misturaria duas perguntas. As outras três métricas de `avaliar` (Sensibilidade, Precisão Média e ROC-AUC) são impressas ao lado, para a escolha ser lida com elas à vista.

A célula abaixo reúne o que as seções anteriores mediram. O Random Forest vem da seção 8.2 (`metricas_rf` e `reconstruido_rf`). O Gradient Boosting vem da busca do #190, que ainda não está neste notebook: quando ela entrar, a seção dela precisa expor `metricas_gb` e `reconstruido_gb` com o mesmo papel, e esta célula passa a compará-los sem alteração. `escolher_melhor_ensemble` deixa fora da disputa qualquer ensemble sem número, ou com `nan`, e sinaliza isso em `comparacao_completa`, para que o texto não escreva que um modelo venceu uma disputa da qual o outro não participou.

In [ ]:
import explicabilidade as expl
importlib.reload(expl)

# Cada ensemble entra com o dicionário de `avaliar` e o pipeline já ajustado no
# treino. `globals().get` porque as seções de origem só definem as variáveis
# quando rodam até o fim; um ensemble ainda não medido entra como None.
METRICAS_ENSEMBLES = {
    "Random Forest": globals().get("metricas_rf"),       # seção 8.2 (#189)
    "Gradient Boosting": globals().get("metricas_gb"),   # busca do #190
}
PIPELINES_ENSEMBLES = {
    "Random Forest": globals().get("reconstruido_rf"),
    "Gradient Boosting": globals().get("reconstruido_gb"),
}

for nome, metricas in METRICAS_ENSEMBLES.items():
    if metricas is None:
        print(f"{nome:18s} sem número medido por avaliar()")
    else:
        print(f"{nome:18s} " + "  ".join(f"{m} {v:.4f}" for m, v in metricas.items()))

if all(m is None for m in METRICAS_ENSEMBLES.values()):
    escolha = None
    pipeline_vencedor = None
    print("\nnenhum ensemble medido: a escolha e a importância ficam pendentes.")
else:
    escolha = expl.escolher_melhor_ensemble(METRICAS_ENSEMBLES, metrica=expl.METRICA_ESCOLHA)
    pipeline_vencedor = PIPELINES_ENSEMBLES[escolha["vencedor"]]
    print(f"\nvencedor pelo {escolha['metrica']}: {escolha['vencedor']}")
    print("comparação entre os dois ensembles completa?", "sim" if escolha["comparacao_completa"] else
          f"não, sem número: {escolha['ausentes']}")
    if escolha["empate"]:
        print("atenção: empate exato no F2, decidido pela ordem da tabela")

O output acima é a evidência do CR01. As primeiras linhas mostram as quatro métricas de `avaliar` de cada ensemble na validação, ou dizem que aquele ensemble ainda não tem número; a linha seguinte diz qual venceu pelo F2.

A linha "comparação completa" é a que decide como o resultado pode ser escrito. Com **sim**, a escolha comparou os dois ensembles e o vencedor pode ser chamado assim. Com **não**, o modelo escolhido é o único que tinha número, e a permutation importance que segue explica esse modelo, mas o texto não pode dizer que ele superou o outro. Esse é o caso enquanto o #190 não entregar a busca do Gradient Boosting, e a tabela da seção 9.5 registra isso.

### 9.2. Cálculo da importância

A configuração do cálculo, que é a evidência do CR02 no código:

| Parâmetro | Valor | Por quê |
| --- | --- | --- |
| `scoring` | `scorer_f2` do #242 | o mesmo critério da busca e da escolha da seção 9.1 |
| `n_repeats` | 10 (`expl.N_REPETICOES`) | com menos, o desvio da queda fica instável demais para separar a terceira da quarta feature; com mais, o custo cresce sem mudar as primeiras posições |
| `random_state` | 42 (`SEMENTE`) | a semente do projeto, a mesma da busca |
| partição | `"validacao"` (`expl.PARTICAO_AVALIACAO`) | a mesma em que a seção 8.2 mediu o número oficial; o teste fica reservado para a comparação final entre duplas (card 18A.1) |

O custo é de uma previsão da partição inteira por feature e por repetição: 14 features do contrato vezes 10 repetições, 140 previsões da validação, mais a de referência. A célula roda o cálculo **duas vezes** e exige que as duas tabelas sejam idênticas, número a número: é o "Como verificar" do card feito pela própria célula, e não deixado para quem lê. Se a sessão estiver apertada, `CONFERIR_REPRODUTIBILIDADE = False` pula a segunda execução; a trava continua coberta por `tests/test_explicabilidade.py`.

In [ ]:
N_REPETICOES = expl.N_REPETICOES
PARTICAO = expl.PARTICAO_AVALIACAO
CONFERIR_REPRODUTIBILIDADE = True

print(f"scoring=scorer_f2 (#242), n_repeats={N_REPETICOES}, random_state={SEMENTE}, partição={PARTICAO!r}")

if escolha is None or pipeline_vencedor is None or scorer_f2 is None:
    importancia = None
    faltando = [nome for nome, objeto in (("ensemble medido (seção 9.1)", escolha),
                                           ("pipeline do vencedor", pipeline_vencedor),
                                           ("#242 (scorer_f2)", scorer_f2)) if objeto is None]
    print("importância não calculada; falta:", ", ".join(faltando))
else:
    # O scoring da importância tem de ser o mesmo objeto que a busca otimizou.
    if busca is not None:
        assert busca.scoring is scorer_f2, "a importância usaria um scoring diferente do da busca"

    importancia = expl.importancia_no_contrato(
        pipeline_vencedor, preparo, scoring=scorer_f2, particao=PARTICAO,
        n_repeats=N_REPETICOES, random_state=SEMENTE,
    )
    if CONFERIR_REPRODUTIBILIDADE:
        de_novo = expl.importancia_no_contrato(
            pipeline_vencedor, preparo, scoring=scorer_f2, particao=PARTICAO,
            n_repeats=N_REPETICOES, random_state=SEMENTE,
        )
        pd.testing.assert_frame_equal(importancia, de_novo)
        print("segunda execução idêntica à primeira: sim")
    print(f"\n{escolha['vencedor']}, partição {PARTICAO}, n={len(preparo['y'][PARTICAO])}")
    display(importancia.round(4))

O output acima começa pela configuração efetivamente usada, que é a mesma da tabela da seção 9.2. Quando o cálculo roda, ele confirma que a segunda execução reproduziu a primeira e mostra a tabela: uma linha por feature **original** do contrato, com a queda média do F2 ao embaralhá-la (`queda_media`), o desvio padrão dessa queda entre as 10 repetições (`queda_desvio`) e a posição no ranking. Nenhuma linha pode ter nome de coluna one-hot, e `TIER_VIAGEM` aparece uma vez só.

Duas leituras exigem cuidado. Uma queda **negativa** não quer dizer que a feature atrapalha: quer dizer que embaralhá-la não piorou o F2 além do ruído, e o modelo, na prática, não a usa. E uma diferença entre duas posições menor do que o desvio delas não separa as duas features. Quando o cálculo não roda, o output lista o que falta, e nenhuma importância substituta é calculada.

### 9.3. Gráfico da importância

O gráfico abaixo é a evidência do CR04. Cada barra é a queda média do F2 ao embaralhar uma feature original, e a barra de erro é o desvio padrão entre as 10 repetições. A feature mais influente fica no alto, e as três primeiras ganham a cor de destaque, porque são elas que seguem para o #192.

In [ ]:
import matplotlib.pyplot as plt
import graficos
graficos.aplicar_tema()

if importancia is None:
    print("gráfico não gerado: a importância da seção 9.2 não foi calculada.")
else:
    figura_importancia = expl.plotar_importancia(
        importancia,
        titulo=f"Permutation importance do {escolha['vencedor']} (validação, {N_REPETICOES} repetições)",
    )
    plt.show()

O output acima, quando o cálculo roda, mostra uma barra por feature original. A linha vertical no zero separa as features de que o modelo depende das que ele poderia perder: uma barra cujo desvio cruza o zero não se distingue do ruído do embaralhamento, e não deve ser lida como fator relevante. Se as três barras de destaque tiverem desvios que se sobrepõem aos da quarta, o ranking do "top 3" é frágil, e isso precisa constar na mensagem ao #192.

### 9.4. Ranking das três features e comparação com a EDA

A EDA (seção 4.2 da documentação) apontou `ATRASO_CHEGADA`, `FAIXA_ATRASO` e `N_TRECHOS` como as features de maior associação individual com a detração. `FAIXA_ATRASO`, porém, **não está no contrato**: ela é a discretização de `ESTATISTICA_ATRASOSAIDA` usada na exploração, e o Feature Set V1 recebe a forma contínua, para não perder a granularidade que a discretização descarta. Por isso a comparação abaixo procura `FAIXA_ATRASO` pelo nome que ela tem no contrato, `ESTATISTICA_ATRASOSAIDA`. Compará-la pelo nome original faria o ranking dizer que ela "sumiu", quando o modelo só a recebe sem discretizar.

A célula também grava `assets/importancia_permutacao.json`, que é o artefato que o #192 lê para saber de quais três features gerar a dependência parcial. O arquivo carrega o ranking inteiro, o ensemble escolhido e a métrica de cada um, a partição, `n_repeats`, `random_state` e o `scoring`. Só tem nomes de feature e quedas agregadas; nenhuma linha de Cliente.

In [ ]:
if importancia is None:
    print("ranking não publicado: a importância da seção 9.2 não foi calculada.")
else:
    topo = expl.features_no_topo(importancia)
    print("três features mais influentes:", ", ".join(f"{i}. {f}" for i, f in enumerate(topo, 1)))
    print()
    display(expl.comparar_com_eda(importancia))

    caminho_importancia = expl.salvar_ranking(
        importancia, escolha, scoring_nome="scorer_f2 (#242)", particao=PARTICAO,
        n_repeats=N_REPETICOES, random_state=SEMENTE,
    )
    print(f"\nranking gravado em assets/{caminho_importancia.name}")

O output acima traz as três features mais influentes na ordem do ranking e uma tabela com as três da EDA: o nome na EDA, o nome no contrato, a posição que cada uma ocupou e se ela ficou entre as três primeiras. A última linha confirma a gravação do JSON que o #192 consome. É a partir desse output que a tabela da seção 9.5 é preenchida e que o ranking é avisado à Fernanda, dona do #192.

### 9.5. Leitura do ranking

| Registro | Valor |
| --- | --- |
| Ensemble explicado | **pendente** (Random Forest, se o #190 não tiver entregado; ver seção 9.1) |
| F2 de `avaliar` do Random Forest (validação) | **pendente do #241 e da execução** |
| F2 de `avaliar` do Gradient Boosting (validação) | **pendente do #190** |
| Comparação entre os dois completa? | **não**, enquanto o #190 estiver pendente |
| `scoring` / `n_repeats` / `random_state` / partição | `scorer_f2` (#242) / 10 / 42 / `validacao` |
| 1ª, 2ª e 3ª features | **pendentes da execução** |
| Posição de `ATRASO_CHEGADA` / `ESTATISTICA_ATRASOSAIDA` (`FAIXA_ATRASO`) / `N_TRECHOS` | **pendentes da execução** |

**Por que a tabela está pendente.** O cálculo depende do vencedor da seção 8, que depende de três MRs ainda abertas (o espaço do #186, o `scorer_f2` do #242 e `avaliar` do #241), e da busca do Gradient Boosting do #190, que ainda não começou. Além disso, não há base da Azul na máquina onde esta seção foi escrita. As células das seções 9.1 a 9.4 foram escritas para rodar sem alteração quando essas peças existirem: rodá-las no Colab sobre a base real preenche esta tabela e grava o JSON de `assets/`. Nenhum número acima foi estimado ou inventado.

**Como comparar com a EDA quando os números existirem.** A expectativa da exploração é que atraso de chegada, atraso de saída e número de trechos liderem. Duas leituras são possíveis, e nenhuma delas é, por si, um problema:

- **O ranking confirma a EDA.** O modelo aprendeu o que a associação individual já mostrava, e o ganho dele está em combinar essas features, como a interação entre tier e atraso da seção 4.2 sugere.
- **O ranking diverge da EDA.** A EDA mede associação de uma feature por vez; a importância mede o quanto o modelo perde sem ela **dado tudo o que ele já tem**. Uma feature forte na EDA pode cair no ranking porque outra já carrega a mesma informação. Esse é o caso esperado para os dois pares redundantes da base: `ATRASO_CHEGADA` e `ESTATISTICA_ATRASOSAIDA` (correlação de 0,664) e `TEMPO_VOO` e `N_TRECHOS` (0,788, por construção). Ao embaralhar uma feature de um par, o modelo recorre à outra, a queda sai menor do que o peso real da informação, e a importância do par fica dividida entre as duas. Por isso, se uma das duas de um par aparecer baixa no ranking, a leitura correta é que o par importa e o modelo escolheu uma delas, e não que aquele fator operacional é irrelevante.

**O que o ranking não diz.** Importância indica **uso pelo modelo, não causa da detração**. Uma feature importante é uma em que o modelo se apoia para separar Detratores de não Detratores nesta base; ela não prova que reduzir aquele fator reduziria a detração, porque o modelo aproveita qualquer associação, inclusive as que vêm de fatores não observados que andam junto com ela. A pergunta causal, a de quanto a detração cai se o atraso cair, não se responde com permutation importance, e o texto da Seção 4.4 (#193) não pode apresentá-la assim. O mesmo vale para o sinal: a importância diz quanto o modelo depende da feature, não em que direção ela empurra a previsão. A direção é o que o #192 mostra, com a dependência parcial das três features acima.

**Nenhuma feature da própria pesquisa.** O ranking só pode conter features do contrato do score pós-viagem. `calcular_importancia` recusa qualquer coluna `NPS_*` ou `SUB_*`, que não existem no momento da predição, e a allowlist do contrato já as exclui na origem (seção 4.3.2.3 da documentação).

**Onde este ranking aparece na documentação.** A configuração da seção 9.2 e a tabela da seção 9.5 são as que a Seção 4.4.4 da documentação (`documents/documentacao.md`, subseção "Explicabilidade por permutation importance") reproduz, junto com a comparação com a EDA e as limitações acima. Ao preencher os campos pendentes com os outputs das seções 9.1 a 9.4, atualize a documentação no mesmo commit.

## 10. Pipeline e linha de base do Gradient Boosting (card 11, #188)

O card 11 foi dividido para liberar o D3 inteiro para a busca de hiperparâmetros do #190: esta seção monta o `Pipeline` que encadeia o `ColumnTransformer` do contrato (`src/matriz.py`) ao `HistGradientBoostingClassifier`, treina a linha de base com hiperparâmetros padrão e mede o resultado com `avaliar` do card 05 (#241).

**A biblioteca é o `HistGradientBoostingClassifier`.** O #185 decidiu por ele em vez do `XGBClassifier`: é o gradient boosting do próprio scikit-learn, já fixado no `requirements.txt`, sem dependência extra para o Colab instalar. A seção 6.2 acima ainda mantém os dois espaços de busca vivos porque o #186 foi escrito antes dessa decisão; a limpeza do espaço descartado é commit do próprio #185, não deste card.

**Sobre a função `avaliar`.** Vem de `src/avaliacao.py`, do card 05 (#241), já em `develop`: a cópia temporária que esta seção carregava enquanto o #241 não tinha chegado foi trocada pelo import, como o card #188 previa. As dependências e a base são as das seções 1 e 2; esta seção só monta a própria partição, na subseção 10.1.

### 10.1. Matriz do contrato

`preparar_matriz` é a única fonte de `X`, `y` e do pré-processador nesta seção, com as mesmas datas de corte do registro de decisão do #127 já usadas em `regressao_logistica.ipynb` — nenhum corte novo é decidido aqui.

In [ ]:
import time

from ensembles import SEMENTE_PADRAO, criar_pipeline_gradient_boosting, medir_linha_de_base

CORTE_VALIDACAO_GB = "2025-07-01"
CORTE_TESTE_GB = "2026-01-01"

inicio = time.perf_counter()
preparo_gb = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO_GB, corte_teste=CORTE_TESTE_GB)
print(f"preparar_matriz: {time.perf_counter() - inicio:.1f}s")

x_treino_gb, y_treino_gb = preparo_gb["x"]["treino"], preparo_gb["y"]["treino"]
x_validacao_gb, y_validacao_gb = preparo_gb["x"]["validacao"], preparo_gb["y"]["validacao"]
print(f"treino: {len(y_treino_gb):,} linhas | validação: {len(y_validacao_gb):,} linhas")

### 10.2. Montagem do pipeline (CR01)

`criar_pipeline_gradient_boosting`, de `src/ensembles.py`, encadeia o `ColumnTransformer` de `preparo_gb["preprocessador"]` — **clonado**, nunca remontado — e o `HistGradientBoostingClassifier`. Sem `hiperparametros`, todo eixo de busca do #190 (`learning_rate`, `max_iter`, `max_leaf_nodes`, `l2_regularization`, `min_samples_leaf`) nasce no padrão da biblioteca; o único valor que a função força é `early_stopping=False`, pela mesma razão documentada em `modelo.HIPERPARAMETROS_CANDIDATO` (#103): o padrão `"auto"` separaria uma fatia aleatória do ajuste que ignora `ID_GOLDENRECORD`. `random_state=SEMENTE_PADRAO` (42) é explícito, não implícito.

In [ ]:
pipeline_gb = criar_pipeline_gradient_boosting(
    preparo_gb["preprocessador"], random_state=SEMENTE_PADRAO,
)
pipeline_gb

**Leitura da saída.** Dois passos nomeados: `preparo` é o `ColumnTransformer` clonado do contrato (ainda não ajustado — o `clone` descarta o estado que `preparar_matriz` já tinha ajustado no treino inteiro), e `modelo` é o `HistGradientBoostingClassifier` com `early_stopping=False` e `random_state=42`, todo o resto no padrão da biblioteca.

### 10.3. Linha de base medida (CR02, CR04)

`medir_linha_de_base` ajusta o pipeline **uma vez** sobre `x_treino_gb`/`y_treino_gb` e pontua sobre `x_validacao_gb`/`y_validacao_gb` — a partição de validação que `preparar_matriz` já separou, não um recorte novo: nenhum `KFold`, `GroupKFold` ou `train_test_split` é instanciado nesta seção nem em `src/ensembles.py` (CR02). `avaliar` é chamada com os três vetores (`y_true`, `y_pred`, `y_proba`), e o dicionário de métricas volta exatamente como ela o devolveu.

In [ ]:
from avaliacao import avaliar

resultado_gb = medir_linha_de_base(
    pipeline_gb, x_treino_gb, y_treino_gb, x_validacao_gb, y_validacao_gb, avaliar,
)
print(f"tempo total (ajuste + predição): {resultado_gb['tempo_total_s']:.1f}s")
print(f"n_treino: {resultado_gb['n_treino']:,} | n_avaliação: {resultado_gb['n_avaliacao']:,}")
resultado_gb["metricas"]

### 10.4. Registro da linha de base (CR04)

| Item | Valor |
|---|---|
| Biblioteca | `HistGradientBoostingClassifier` (scikit-learn), decidida pelo #185 |
| Versão | `scikit-learn 1.9.1` (`requirements.txt`, chore #206) |
| `random_state` | `42` (`ensembles.SEMENTE_PADRAO`) |
| Partição avaliada | validação (48.301 linhas), treino com 341.962 linhas |
| F2 | 0,2936 |
| Sensibilidade | 0,2556 |
| Precisão Média | 0,5176 |
| ROC-AUC | 0,7340 |
| Tempo total (ajuste + predição) | 8,9s |

Essa é a linha de base do card: o desempenho do Gradient Boosting **antes** de qualquer busca de hiperparâmetro, com o pré-processador do contrato e `early_stopping=False` (a única troca em relação ao padrão da biblioteca, pela razão de vazamento por Cliente da seção 10.2). É contra estes quatro números que a busca aleatória do #190 (`ESPACO_GRADIENT_BOOSTING_HISTGB`, seção 6.2.1 acima) precisa mostrar ganho; sem essa referência, qualquer resultado da busca pareceria bom por si só.

## 11. Busca aleatória do Gradient Boosting (#190)

O #188 deixou o `HistGradientBoostingClassifier` montado sobre o pré-processador do contrato, com os hiperparâmetros padrão da biblioteca, e registrou a linha de base da seção 10.4. Esta seção entrega esse mesmo pipeline a uma `RandomizedSearchCV`, que varre o espaço `ESPACO_GRADIENT_BOOSTING_HISTGB` do #186 (seção 6.2.1) e grava os hiperparâmetros vencedores num JSON versionado. A partir desse JSON, `ensembles.melhor_gradient_boosting()` remonta o vencedor para a dupla de Métricas e Decisões.

Três travas valem para tudo o que segue:

1. **A validação é agrupada por Cliente.** O mesmo `ID_GOLDENRECORD` aparece em vários voos. Os folds vêm de `validacao.criar_folds`, o splitter agrupado do contrato, e são conferidos por `validacao.conferir_folds` antes da busca. O `fit` recebe `groups=groups`. `cv` inteiro é recusado por `criar_busca_gradient_boosting`, pelo mesmo motivo do #189: num classificador, o scikit-learn converteria o inteiro num particionador estratificado que ignora o Cliente, e o vencedor seria o que mais memoriza pessoas.
2. **`n_iter = 40` e `random_state = 42` (CR01).** Quarenta é o mínimo do card, e a função recusa menos. A semente `42` (`SEMENTE_PADRAO`) fixa ao mesmo tempo o sorteio das 40 combinações e a semente de cada `HistGradientBoostingClassifier`, então duas execuções desta seção sorteiam as mesmas combinações e elegem o mesmo vencedor.
3. **O scoring é o oficial da dupla de Métricas e Decisões.** É o `scorer_f2` do #242 (F2 com `beta=2`, Detrator como classe positiva), o critério de busca que o protocolo do #238 fixou para as quatro duplas. A partição de teste não é tocada: a busca recebe só o treino, e a reconstrução da seção 11.3 é medida na validação, a mesma partição da linha de base.

### 11.1. Montagem da busca (CR01, CR02)

A célula abaixo materializa os cinco folds agrupados por Cliente sobre o treino cru, confere que nenhum Cliente vaza entre ajuste e validação e monta a `RandomizedSearchCV` com `busca_gradient_boosting.criar_busca_gradient_boosting`. Nada é ajustado ainda. `n_jobs=-1` distribui os ajustes inteiros (uma combinação num fold) entre os núcleos da sessão. A ordem em que eles terminam não muda nenhum número.

In [ ]:
import busca_gradient_boosting as busca_gb
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB
from scorer_f2 import scorer_f2
from validacao import N_FOLDS, conferir_folds, criar_folds

N_ITER = 40
RANDOM_STATE_BUSCA = SEMENTE_PADRAO  # 42

groups = preparo_gb["grupos"]["treino"]
folds_gb = criar_folds(x_treino_gb, groups)
conferir_folds(folds_gb, groups)
print(f"folds do contrato: {len(folds_gb)}, conferidos contra o Cliente")

busca = busca_gb.criar_busca_gradient_boosting(
    preparo_gb["preprocessador"], ESPACO_GRADIENT_BOOSTING_HISTGB, folds_gb, scorer_f2,
    n_iter=N_ITER, random_state=RANDOM_STATE_BUSCA, n_jobs=-1,
)
print(f"n_iter={busca.n_iter} | random_state={busca.random_state} | folds={len(busca.cv)} | scoring={busca.scoring}")
print(f"ajustes previstos: {N_ITER} x {N_FOLDS} + refit = {N_ITER * N_FOLDS + 1}")

**Leitura da saída.** A primeira linha confirma que os cinco folds saíram de `criar_folds` e passaram por `conferir_folds`: nenhum Cliente aparece em duas validações, nem no ajuste e na validação do mesmo fold. A segunda mostra a configuração que o CR01 pede, `n_iter=40` e `random_state=42`, os cinco folds do contrato no lugar de `cv` e o scorer F2 do #242 (`make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)`). A terceira dá o custo da busca: 201 ajustes, 40 combinações vezes 5 folds mais o reajuste do vencedor no treino inteiro. Com os 8,9 s da linha de base (seção 10.4) por ajuste, a conta sequencial fica perto de 30 minutos. O paralelismo entre processos a reduz.

### 11.2. Execução da busca e artefatos versionados

`busca_gradient_boosting.executar_busca` confere os folds de novo, chama `busca.fit(x_treino, y_treino, groups=groups)` sobre a matriz **crua** do treino (quem transforma é o pipeline, dentro de cada fold) e mede o tempo total. `salvar_resultados` grava dois arquivos em `assets/`, os únicos desta seção que vão para o git:

- `hiperparametros_gradient_boosting.json`: os hiperparâmetros vencedores, o `random_state`, o `n_iter`, o número de folds, o melhor F2 médio e o tempo total. É o arquivo que `ensembles.melhor_gradient_boosting()` lê;
- `cv_resultados_gradient_boosting.json`: uma linha por combinação, só com parâmetros, F2 médio, desvio entre folds, posição no ranking e tempo médio de ajuste. Nenhum dado de Cliente entra.

Os dois são JSON, e não CSV, porque o `.gitignore` do projeto proíbe `*.csv` por compromisso com o parceiro. O modelo ajustado não é versionado, por tamanho.

In [ ]:
relato_busca = busca_gb.executar_busca(busca, x_treino_gb, y_treino_gb, groups)
caminho_hp, caminho_resumo = busca_gb.salvar_resultados(busca, relato_busca, random_state=RANDOM_STATE_BUSCA)

vencedores_gb = busca_gb.hiperparametros_vencedores(busca)
print("hiperparâmetros vencedores:")
for nome, valor in vencedores_gb.items():
    print(f"  {nome:18s} {valor}")
print(f"\nmelhor F2 médio nos folds: {relato_busca['melhor_score_medio']:.4f}")
print(f"tempo total da busca: {relato_busca['tempo_total_s']:.0f} s "
      f"({relato_busca['tempo_total_s'] / 60:.1f} min), {relato_busca['n_ajustes']} ajustes")
print("no limite do intervalo:", busca_gb.parametros_no_limite(vencedores_gb, ESPACO_GRADIENT_BOOSTING_HISTGB) or "nenhum")
print("\ngravados:", caminho_hp.name, "e", caminho_resumo.name)
busca_gb.resumir_cv_results(busca).head(10)

**Leitura da saída.** O bloco impresso traz os seis hiperparâmetros vencedores, o melhor F2 médio nos cinco folds, o tempo total e o número de ajustes. Esse número precisa ser 201: menos do que isso significaria um fold pulado ou uma combinação que falhou. A linha "no limite do intervalo" lista os vencedores que caíram na borda do espaço do #186. Um vencedor na borda diz que a busca queria ir além dela, e por isso é comentado na tabela da seção 11.4. A tabela no fim mostra as dez melhores combinações, com o desvio entre folds (`std_test_score`) ao lado da média. Uma diferença de F2 menor que esse desvio não separa dois candidatos.

### 11.3. Reconstrução pelo JSON e métrica oficial (CR04, CR05)

O F2 médio da seção 11.2 é o critério da busca, não o número que vai para a tabela comparativa. O número oficial é o de `avaliar`, do #241, e ele precisa sair de um modelo que qualquer pessoa consiga remontar sem esta sessão aberta. Por isso a célula abaixo não usa o `best_estimator_`. Ela chama `ensembles.melhor_gradient_boosting()`, a mesma função que a dupla de Métricas e Decisões vai importar, e essa função lê o JSON gravado em `assets/` e devolve o pipeline vencedor **não ajustado**, com o `random_state` do JSON.

O pipeline reconstruído passa por `medir_linha_de_base`, a mesma medição da seção 10.3: ajuste no treino, previsão na validação e `avaliar(y_true, y_pred, y_proba)`. Linha de base e vencedor ficam assim comparáveis número a número. `avaliar` agora vem de `src/avaliacao.py`, já em `develop`, e não mais de uma cópia temporária. Por fim, a célula mede também o `best_estimator_` da busca desta sessão e exige que as duas medições sejam iguais.

In [ ]:
from avaliacao import avaliar
from ensembles import PASSO_MODELO, melhor_gradient_boosting
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted

melhor_gb = melhor_gradient_boosting(preparo_gb["preprocessador"])
try:
    check_is_fitted(melhor_gb.named_steps[PASSO_MODELO])
    print("pipeline reconstruído já ajustado: não deveria")
except NotFittedError:
    print("pipeline reconstruído chega não ajustado")
print("random_state do reconstruído:", melhor_gb.named_steps[PASSO_MODELO].random_state)

resultado_melhor = medir_linha_de_base(melhor_gb, x_treino_gb, y_treino_gb, x_validacao_gb, y_validacao_gb, avaliar)
metricas_melhor = resultado_melhor["metricas"]

melhor_busca = busca.best_estimator_
metricas_busca = avaliar(y_validacao_gb, melhor_busca.predict(x_validacao_gb), melhor_busca.predict_proba(x_validacao_gb)[:, 1])
assert metricas_busca == metricas_melhor, "o JSON não reconstrói o vencedor da busca"

comparacao = pd.DataFrame({
    "linha de base (#188)": resultado_gb["metricas"],
    "vencedor da busca (#190)": metricas_melhor,
})
comparacao["diferença"] = comparacao["vencedor da busca (#190)"] - comparacao["linha de base (#188)"]
print(f"tempo do reconstruído (ajuste + predição): {resultado_melhor['tempo_total_s']:.1f} s")
print("idêntico ao best_estimator_ da busca: sim")
comparacao.round(4)

**Leitura da saída.** As duas primeiras linhas confirmam o CR04 sobre o objeto real: `melhor_gradient_boosting()` devolve o pipeline **sem ajuste**, com `random_state=42`, o mesmo do JSON. A linha "idêntico ao best_estimator_" só aparece se as quatro métricas do reconstruído forem exatamente iguais às do vencedor reajustado pela busca. Essa igualdade faz do JSON versionado uma descrição completa do vencedor. A tabela põe lado a lado as métricas de `avaliar` da linha de base da seção 10.4 e as do vencedor, na mesma partição de validação (48.301 linhas), e é dela que saem os números registrados na seção 11.4 (CR05).

### 11.4. Espaço de busca, vencedores e registro (CR01, CR03, CR05)

**Espaço do #186 lado a lado com o vencedor.** A coluna "No limite?" usa `parametros_no_limite`: um inteiro conta como borda só no piso ou no teto exatos, e um contínuo conta quando fica a menos de 5% da amplitude de um dos extremos (na escala log para o `learning_rate`).

| Hiperparâmetro | Espaço do #186 | Vencedor | No limite? |
| --- | --- | --- | --- |
| `learning_rate` | `loguniform` de 0,01 a 0,3 | 0,0498 | não |
| `max_leaf_nodes` | `randint` de 15 a 127 | 61 | não |
| `max_iter` | `randint` de 100 a 600 | 200 | não |
| `l2_regularization` | `uniform` de 0,0 a 2,0 | 1,7744 | não |
| `min_samples_leaf` | `randint` de 20 a 200 | 150 | não |
| `class_weight` | `None` ou `"balanced"` | `"balanced"` | não se aplica |
| `early_stopping` | fora do espaço, fixo em `False` pelo #188 | `False` | não se aplica |

**Registro da busca.**

| Item | Valor |
| --- | --- |
| `n_iter` | **40** (mínimo do card, recusado abaixo disso) |
| `random_state` | **42** (`SEMENTE_PADRAO`), no sorteio e no estimador |
| `cv` | 5 folds de `validacao.criar_folds`, agrupados por `ID_GOLDENRECORD`; `fit(..., groups=groups)` |
| `scoring` | `scorer_f2` do #242 (F2, `beta=2`, Detrator positivo) |
| Ajustes | 201 (40 × 5 + refit) |
| F2 médio na validação cruzada (5 folds) do vencedor | 0,5318 (desvio entre folds 0,0035) |
| Tempo total registrado | 7147 s de parede, `n_jobs=-1` em 12 núcleos, execução local (ver "Sobre o tempo" abaixo) |
| Hiperparâmetros versionados | `assets/hiperparametros_gradient_boosting.json` |
| Resumo das 40 combinações | `assets/cv_resultados_gradient_boosting.json` |

**Métrica do estimador reconstruído em `avaliar` (CR05).** Estes são os números que a célula da seção 11.3 imprime para `melhor_gradient_boosting()` ajustado no treino e medido na validação, ao lado da linha de base da seção 10.4.

| Métrica (`avaliar`, #241) | Linha de base (#188) | Vencedor reconstruído (#190) | Diferença |
| --- | --- | --- | --- |
| F2 | 0,2936 | 0,4974 | +0,2038 |
| Sensibilidade | 0,2556 | 0,5068 | +0,2511 |
| Precisão Média | 0,5176 | 0,5174 | −0,0002 |
| ROC-AUC | 0,7340 | 0,7330 | −0,0010 |

**Leitura.** O ganho de F2 na validação, de 0,2936 para 0,4974, vem quase todo de um eixo só, `class_weight`. As 21 combinações com `"balanced"` ficaram entre 0,512 e 0,532 de F2 médio na validação cruzada, e as 19 sem reponderação ficaram entre 0,311 e 0,330, sem nenhuma sobreposição. Reponderar empurra a probabilidade predita para cima, e o corte padrão de 0,5 do `predict` passa a apontar mais Detratores. É isso que o F2, que trata o recall como duas vezes mais importante que a precisão (Seção 4.4.1 da documentação), premia: a Sensibilidade praticamente dobra (0,2556 para 0,5068). Precisão Média e ROC-AUC, que só dependem da ordenação do score, ficam onde estavam (−0,0002 e −0,0010).

Ou seja, a busca não produziu um modelo que **ordena** melhor os Clientes. Ela produziu um que **corta** num ponto mais favorável ao recall. Isso importa para a dupla de Métricas e Decisões. O limiar operacional por capacidade de contato (seção 4.3.2, card #247) vai redefinir esse corte de qualquer forma, e a comparação entre os quatro modelos deve ler Precisão Média e ROC-AUC ao lado do F2. Vale também a ressalva do #103: com `"balanced"`, a probabilidade predita deixa de refletir a frequência observada de 20,44% de Detratores e não pode ser lida diretamente como risco.

Entre as combinações balanceadas, as 13 melhores estão a menos de um desvio entre folds (0,0035) do vencedor. Os eixos numéricos quase não separam candidatos, e como nenhum vencedor ficou no limite do espaço, o intervalo do #186 não precisa ser ampliado. A linha de base reexecutada nesta sessão reproduziu exatamente os quatro números da seção 10.4.

**Sobre o tempo.** A busca rodou localmente, com `n_jobs=-1` em 12 núcleos, e não no Colab. No meio da execução a máquina ficou parada por cerca de 75 minutos, então os 7.147 s registrados superestimam o custo: o tempo efetivo de busca ficou perto de 45 minutos. Como referência de orçamento, vale o tempo médio de ajuste por combinação, que está em `cv_resultados_gradient_boosting.json` (93 s no vencedor, até 260 s nas combinações com `max_iter` acima de 500).

### 11.5. Como a dupla de Métricas e Decisões consome o vencedor

```python
from ensembles import melhor_gradient_boosting
from matriz import preparar_matriz

preparo = preparar_matriz(df, corte_validacao="2025-07-01", corte_teste="2026-01-01")
pipeline = melhor_gradient_boosting(preparo["preprocessador"])   # não ajustado, random_state=42
pipeline.fit(preparo["x"]["treino"], preparo["y"]["treino"])
```

A função não recebe hiperparâmetro nem semente: os dois vêm do JSON versionado, e é isso que garante que qualquer pessoa remonte o mesmo modelo. As travas que protegem a entrega rodam com `pytest tests/test_ensembles.py tests/test_busca_gradient_boosting.py -v`.

**Depende de:** #186 (espaço), #188 (pipeline e linha de base), #241 (`avaliar`) e #242 (`scorer_f2`). **Bloqueia:** #191, #192 e #194, que partem de `melhor_gradient_boosting()`.

## 12. Dependências e próximos passos

**Incorpora:** o #186 (espaços de busca), que passou a viver na seção 6 deste mesmo notebook quando as duas frentes se encontraram em `src/ensembles.py`.

**Depende de:** card 01 (contrato de dados, Kaylan e Pedro) — `x`, `y`, `grupos` e o `ColumnTransformer`, consumidos na seção 2 via `matriz.preparar_matriz`.

**Bloqueia:** #187 e #188, que só podem dimensionar a busca depois da decisão da seção 5. A seção 7 (#187) depende ainda do card 05 (#241), dono de `avaliar`, e bloqueia a busca do Random Forest (#189). A seção 8 (#189) depende do #186 (espaço de busca), do #242 (`scorer_f2`) e do #241 (`avaliar`), e bloqueia o #191 e o #193. A seção 9 (#191) depende da seção 8 e da busca do Gradient Boosting (#190), e bloqueia o #192, que gera a dependência parcial das três features do ranking, e o #193, que escreve a subseção da Seção 4.4. A seção 10 (#188) depende do #241 (`avaliar`) e bloqueia a busca do Gradient Boosting (#190). A seção 11 (#190) depende da seção 10, do #242 (`scorer_f2`) e do #241, e bloqueia o #191, o #192 e o #194.

**Próximo passo obrigatório:** rodar este notebook em sessão nova do Colab, sobre a base analítica real, e substituir a tabela da seção 3.1 e os campos pendentes da seção 7.2 pelos outputs medidos lá. Os campos pendentes da seção 9.5 entram na mesma execução, e as travas dela estão em `tests/test_explicabilidade.py`. As travas que protegem a validade da medição estão em `tests/test_ensembles.py` e rodam com `pytest tests/test_ensembles.py -v`.